# 3. Imputation of Missing Data 

In line with Step 3 of the OECD Handbook, this section addresses missing values to ensure a complete dataset before index construction. Leaving missing entries unresolved would distort comparisons and reduce the reliability of the final composite index.

### Core Methodology:
* **Temporal Compaction (Most Recent Value):** To reduce reporting lag issues, the 2016–2026 timeline is converted into a cross-sectional dataset. For each country-indicator pair, the pipeline selects the most recent available value by scanning backward from 2026.

* **Peer-Group Imputation (Cross-Sectional Fill):** Some countries contain fully missing indicators across the entire timeline, particularly for advanced financial market variables. In these cases, missing values are estimated using peer-group averages.

* **Economic Stratification:** Instead of applying a global mean, missing values are filled using the average of the country’s World Bank income group (High, Low, Lower-Middle, or Upper-Middle Income). This preserves regional and economic comparability while reducing imputation bias.

### Post-Imputation Quality Diagnostics:
* **Impact Assessment:** The percentage of imputed values is calculated for each variable to evaluate overall dataset reliability after imputation.

* **Outlier Profile Review:** An Interquartile Range (IQR) analysis is applied to identify extreme values before normalization and index aggregation.

In [13]:
import os
import numpy as np
import pandas as pd

filtered_data_path = "../data/worldbank_filtered_data.csv"
metadata_path = "../data/worldbank_metadata.csv"

In [15]:
if not os.path.exists(filtered_data_path):
    print(f"'{filtered_data_path}' not found!")

else:
    df_filtered_load = pd.read_csv(filtered_data_path)

    # Create a latest-value snapshot from the timeline dataset
    df_compaction = df_filtered_load.copy()

    year_cols = sorted(
        [col for col in df_compaction.columns if col.isdigit()],
        reverse=True
    )

    # Select the most recent available value for each row
    def get_latest_value(row):
        for year in year_cols:
            if pd.notnull(row[year]):
                return row[year]
        return np.nan

    df_compaction['Latest_Value'] = df_compaction.apply(get_latest_value, axis=1)

    # Convert the dataset into a country-level matrix
    df_snapshot = df_compaction.pivot(
        index=['Country Name', 'Country Code'],
        columns='Variable',
        values='Latest_Value'
    ).reset_index()